In [0]:
# Create widgets
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("load_type", "")
dbutils.widgets.text("merge_key", "")

# Get widget values into variables
landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name = dbutils.widgets.get("table_name")
load_type = dbutils.widgets.get("load_type").strip().upper()
merge_key = dbutils.widgets.get("merge_key")

print("="*60)
print("STEP 1: Parameter Initialization")
print("="*60)
print(f"  landing_timestamp : {landing_timestamp}")
print(f"  table_name        : {table_name}")
print(f"  load_type         : {load_type}")
print(f"  merge_key         : {merge_key}")
print("="*60)

In [0]:
bronze_df = spark.read\
                .option("inferschema", "true")\
                .table(f"adb_qnxthealth.bronze.{table_name}")\
                .filter(f"landing_timestamp='{landing_timestamp}'")
#display(bronze_df.limit(5))

In [0]:
from pyspark.sql.functions import current_timestamp

updated_bronze_df = bronze_df.withColumn("insert_timestamp", current_timestamp())\
                            .withColumn("update_timestamp", current_timestamp())


In [0]:
from delta.tables import DeltaTable

silver_table = f"adb_qnxthealth.silver.{table_name}"

print("="*60)
print("STEP 4: Writing to Silver Table")
print("="*60)
print(f"  Target table : {silver_table}")
print(f"  Load type    : {load_type}")

# Early exit if no records to process
if updated_bronze_df.count() == 0:
    print("  No records found in updated_bronze_df. Skipping write.")
    print("="*60)
    dbutils.notebook.exit("0")

if load_type == "FULL":
    print(f"  Strategy     : FULL overwrite")
    print(f"  Write command: updated_bronze_df.write.mode('overwrite').saveAsTable('{silver_table}')")
    updated_bronze_df.write.mode("overwrite").saveAsTable(silver_table)
    print(f"  Write complete! Table '{silver_table}' overwritten successfully.")

elif load_type == "APPEND":
    print(f"  Strategy     : APPEND")
    print(f"  Write command: updated_bronze_df.write.mode('append').saveAsTable('{silver_table}')")
    updated_bronze_df.write.mode("append").saveAsTable(silver_table)
    print(f"  Write complete! Data appended to '{silver_table}' successfully.")

elif load_type == "MERGE":
    # Support multiple merge keys (comma-separated)
    merge_keys = [k.strip() for k in merge_key.split(",") if k.strip()]
    if not merge_keys:
        raise ValueError("merge_key must be provided when load_type is 'MERGE'.")
    merge_condition = " AND ".join([f"target.'{k}' = source.'{k}'" for k in merge_keys])

    print(f"  Strategy       : MERGE (upsert)")
    print(f"  Merge keys     : {merge_keys}")
    print(f"  Merge condition: {merge_condition}")

    table_exists = spark.catalog.tableExists(silver_table)
    print(f"  Table exists   : {table_exists}")

    if table_exists:
        delta_table = DeltaTable.forName(spark, silver_table)
        # Preserve original insert_timestamp on updates; update all other columns
        update_cols = {c: f"source.{c}" for c in updated_bronze_df.columns if c != "insert_timestamp"}

        print(f"  Update columns : {list(update_cols.keys())}")
        print(f"  Merge query:")
        print(f"    MERGE INTO {silver_table} AS target")
        print(f"    USING updated_bronze_df AS source")
        print(f"    ON {merge_condition}")
        print(f"    WHEN MATCHED THEN UPDATE SET {', '.join([f'{k} = source.{k}' for k in update_cols])}")
        print(f"    WHEN NOT MATCHED THEN INSERT ALL")

        delta_table.alias("target").merge(
            updated_bronze_df.alias("source"),
            merge_condition
        ).whenMatchedUpdate(set=update_cols) \
         .whenNotMatchedInsertAll() \
         .execute()
        print(f"  Merge complete! Data merged into '{silver_table}' successfully.")
    else:
        # First load - table doesn't exist yet, create it
        print(f"  Table does not exist. Creating with initial load.")
        print(f"  Write command: updated_bronze_df.write.saveAsTable('{silver_table}')")
        updated_bronze_df.write.saveAsTable(silver_table)
        print(f"  Initial load complete! Table '{silver_table}' created successfully.")

else:
    raise ValueError(f"Unsupported load_type: '{load_type}'. Expected 'FULL', 'APPEND', or 'MERGE'.")

print("="*60)

In [0]:
record_count = updated_bronze_df.count()
print(f"  Records count : {record_count}")
dbutils.notebook.exit(str(record_count))